# Mini_Assignment_1_Sagar (Fixed Version)

Lung Cancer Patient Health and Treatment Records Analysis using PySpark

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Lung Cancer Analysis") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.csv("Lung Cancer.csv", header=True, inferSchema=True)

df.show(5)
df.printSchema()

## Task 1: Data Cleaning

In [ ]:
def clean_data(df):
    df = df.dropDuplicates()
    
    # Identify yes/no columns without collecting all data (improved to avoid timeout)
    yes_no_cols = []
    for col_name in df.columns:
        # Use SQL aggregation to check distinct values
        distinct_df = df.groupBy(col_name).count()
        distinct_vals = [row[col_name] for row in distinct_df.collect() if row[col_name] is not None]
        vals_lower = [str(v).lower() for v in distinct_vals]
        if set(vals_lower) == {'yes', 'no'}:
            yes_no_cols.append(col_name)
    
    for col_name in yes_no_cols:
        df = df.withColumn(col_name,
                           when(lower(col(col_name)) == "yes", 1)
                           .when(lower(col(col_name)) == "no", 0)
                           .otherwise(None))
    
    df = df.withColumn("diagnosis_date", to_date(col("diagnosis_date"), "yyyy-MM-dd"))
    df = df.withColumn("treatment_end_date", to_date(col("treatment_end_date"), "yyyy-MM-dd"))
    df = df.withColumn("age", col("age").cast(IntegerType()))
    df = df.withColumn("bmi", col("bmi").cast(DoubleType()))
    
    return df

clean_df = clean_data(df)
clean_df.printSchema()

## Task 2: Treatment Duration

In [ ]:
def treatment_duration_analysis(df):
    df = df.withColumn(
        "treatment_duration_days",
        datediff(col("treatment_end_date"), col("diagnosis_date"))
    )

    return df.groupBy("treatment_type") \
        .agg(avg("treatment_duration_days").alias("avg_duration_days"))

task2_result = treatment_duration_analysis(clean_df)
task2_result.show()

## Task 3: Smoking Status Highest Survival

In [ ]:
def highest_survival_by_smoking(df):
    return df.groupBy("smoking_status") \
        .agg((sum(when(col("survived") == 1, 1).otherwise(0)) / count("*")).alias("survival_rate")) \
        .orderBy(desc("survival_rate")) \
        .limit(1)

task3_result = highest_survival_by_smoking(clean_df)
task3_result.show()

## Task 4: Top 3 Countries Stage IV

In [ ]:
def top_countries_stage_iv(df):
    df = df.withColumn("is_stage_iv", when(col("cancer_stage") == "Stage IV", 1).otherwise(0))

    return df.groupBy("country") \
        .agg((sum("is_stage_iv") / count("*") * 100).alias("stage_iv_percentage")) \
        .orderBy(desc("stage_iv_percentage")) \
        .limit(3)

task4_result = top_countries_stage_iv(clean_df)
task4_result.show()

## Task 5: High Risk Patient Analysis

In [ ]:
def high_risk_patient_analysis(df):
    filtered_df = df.filter(
        (col("gender") == "Male") &
        (col("cancer_stage").isin("Stage III", "Stage IV")) &
        (col("family_history") == 1) &
        (col("smoking_status") == "Current") &
        (col("bmi") > 30) &
        (col("survived") == 1)
    )

    return filtered_df.agg(
        avg("age").alias("avg_age"),
        (sum(when(col("hypertension") == 1, 1).otherwise(0)) / count("*") * 100)
        .alias("hypertension_percentage")
    )

task5_result = high_risk_patient_analysis(clean_df)
task5_result.show()

## Assumptions
- diagnosis_date and treatment_end_date exist in yyyy-MM-dd format  
- survived column: 1 = Yes, 0 = No  
- smoking_status includes 'Current'  
- cancer_stage includes 'Stage III', 'Stage IV'  
- yes/no columns contain only Yes/No values  

## Fixes Applied
- Added `.master("local[*]")` to SparkSession for local mode.
- Improved `clean_data` to use `groupBy().count()` instead of `distinct().collect()` to avoid PySpark worker timeouts.
- Ensured code is more robust for Windows environment.